# PKG TM customer x month — diagnostics before the rebuild

Read-only. Two questions from the first full build (2026-11-09):

1. **Value coverage**: the customer table holds only about 40-50% of the source's billed amount and investable balance. Are the missing dollars on accounts without a power id, or does the source total double-count parent rows of the billing hierarchy? And can accounts without a power id be linked (current customer key in the same snapshot, customer dimension)?
2. **DHE doubling**: DHE about doubles in 2025-08 and 2026-03. Does the month-end date hold two full customer-fact loads with different cust_dim_keys? Compares the current rule (latest version per cust_dim_key) with the proposed one (latest load per power id).

In [ ]:
# ============================================================
# 1. Setup and config
# ============================================================
import os, re, html, warnings, datetime as _dt
warnings.filterwarnings("ignore", category=ResourceWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore")

# --- NumPy alias shim (old pyspark code paths use removed aliases) ---
import numpy as np
for _a, _t in (("object0", object), ("bool8", np.bool_), ("int0", np.intp), ("uint0", np.uintp),
               ("bool", bool), ("object", object)):
    if not hasattr(np, _a):
        setattr(np, _a, _t)

# --- pandas shim: PySpark 3.3 createDataFrame calls iteritems (removed in pandas 2.0) ---
import pandas as pd
if not hasattr(pd.DataFrame, "iteritems"):
    pd.DataFrame.iteritems = pd.DataFrame.items
if not hasattr(pd.Series, "iteritems"):
    pd.Series.iteritems = pd.Series.items

from IPython.display import display, HTML
from pyspark.sql import SparkSession, Window, functions as F, types as T
from pyspark import StorageLevel

spark = (SparkSession.builder
         .appName("pkg_tm_cust_month_diagnostics")
         .config("spark.ui.showConsoleProgress", "false")   # no stage progress bars in cell outputs
         .enableHiveSupport()
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
try:  # silence Atlas lineage hook + harmless executor-release noise (log4j2 on Spark 3.3)
    _L = spark._jvm.org.apache.logging.log4j
    for n in ("org.apache.atlas", "com.hortonworks.spark.atlas", "com.cloudera.spark.lineage",
              "org.apache.kafka", "org.apache.spark.sql.catalyst.util.package",
              "org.apache.spark.network", "io.netty"):
        _L.core.config.Configurator.setLevel(n, _L.Level.OFF)
except Exception:
    pass
# Unparseable date strings -> NULL instead of SparkUpgradeException (we probe many formats)
spark.conf.set("spark.sql.legacy.timeParserPolicy", "CORRECTED")

# ---------------- config ----------------
OFS_BAL     = "clhhd01p_clh.cdm_tm_ofs_bal_analysis_fact_v"
CUST_FACT   = "clhhd01p_clh.cdm_customer_fact_v"
CUST_DIM    = "clhhd01p_clh.cdm_customer_dim_v"
VALUE_MONTHS = ["2025-06", "2026-07", "2026-08", "2026-09"]   # analysis months (OFS snapshot = month + 1)
DHE_MONTHS   = ["2025-07", "2025-08", "2026-02", "2026-03", "2026-08", "2026-09"]
SHIFT       = 1
TOP_N       = 15
OUT_DIR     = "../metrics/tm_cust_month/diagnostics"
os.makedirs(OUT_DIR, exist_ok=True)

# ---------------- helpers ----------------
CSS = """<style>
.cx {font-family: Arial, sans-serif; font-size: 12px;}
.cx h3 {margin: 14px 0 4px 0; font-size: 14px;}
.cx h4 {margin: 10px 0 3px 0; font-size: 13px; color: #1f3b63;}
.cx table {border-collapse: collapse; margin: 2px 0 8px 0;}
.cx th {background: #e8eef6; border: 1px solid #c8d2e0; padding: 2px 6px; text-align: left; white-space: nowrap;}
.cx td {border: 1px solid #dde3ec; padding: 2px 6px; vertical-align: top; max-width: 420px; overflow-wrap: anywhere;}
.cx .note {color: #555; margin: 2px 0;}
.cx .warn {color: #a33; font-weight: bold;}
.cx pre {background: #f6f8fa; border: 1px solid #dde3ec; padding: 6px; white-space: pre-wrap; font-size: 11px; max-width: 1400px;}
</style>"""

def esc(x):
    return html.escape("" if x is None else str(x))

def tbl(pdf, max_rows=None):
    """pandas DataFrame -> compact HTML table (values escaped)."""
    if pdf is None or len(pdf) == 0:
        return "<p class='note'>(no rows)</p>"
    if max_rows is not None:
        pdf = pdf.head(max_rows)
    head = "".join(f"<th>{esc(c)}</th>" for c in pdf.columns)
    body = "".join("<tr>" + "".join(f"<td>{esc(v)}</td>" for v in row) + "</tr>"
                   for row in pdf.astype(object).where(pdf.notna(), "").itertuples(index=False))
    return f"<table><tr>{head}</tr>{body}</table>"

def show(parts):
    """One consolidated HTML output per cell; tex2jax_ignore keeps any dollar sign from turning into math."""
    display(HTML(CSS + "<div class='cx tex2jax_ignore mathjax_ignore'>" + "".join(parts) + "</div>"))

def fnum(x):
    if x is None:
        return ""
    try:
        x = float(x)
    except Exception:
        return str(x)
    if np.isnan(x):
        return ""
    a = abs(x)
    for d, s in ((1e12, "T"), (1e9, "B"), (1e6, "M"), (1e3, "K")):
        if a >= d:
            return f"{x / d:.2f}{s}"
    if x == int(x):
        return str(int(x))
    return f"{x:.4g}"

def fpct(x, digits=1):
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return ""
    return f"{100 * float(x):.{digits}f}%"

def qcol(c):
    return F.col(f"`{c}`")

NUMERIC_T = (T.ByteType, T.ShortType, T.IntegerType, T.LongType, T.FloatType, T.DoubleType, T.DecimalType)
COMPLEX_T = (T.ArrayType, T.MapType, T.StructType, T.BinaryType)
DATE_NAME = re.compile(r"(_dt$|_dte$|_date$|^date_|_date_|_ts$|_tmstmp|timestamp|^dt_|_dt_|asof|as_of|snap|_month$|_mo$|_yrmo|yyyymm|period)")

def kind_of(dtype):
    if isinstance(dtype, (T.DateType, T.TimestampType)): return "date"
    if isinstance(dtype, T.BooleanType): return "bool"
    if isinstance(dtype, NUMERIC_T): return "num"
    if isinstance(dtype, COMPLEX_T): return "complex"
    return "str"

def is_date_like(name, dtype):
    k = kind_of(dtype)
    if k == "date":
        return True
    return k in ("str", "num") and bool(DATE_NAME.search(name.lower())) \
        and not isinstance(dtype, (T.FloatType, T.DoubleType))

def date_expr(c, dtype):
    """Parse a date-like column of any common storage format to DATE (NULL if unparseable)."""
    col = qcol(c)
    if isinstance(dtype, T.DateType):
        return col
    if isinstance(dtype, T.TimestampType):
        return F.to_date(col)
    s = F.trim(col.cast("string"))
    return (F.when(s.rlike(r"^\d{4}-\d{2}-\d{2}"), F.to_date(F.substring(s, 1, 10), "yyyy-MM-dd"))
             .when(s.rlike(r"^\d{8}$"), F.to_date(s, "yyyyMMdd"))
             .when(s.rlike(r"^\d{1,2}/\d{1,2}/\d{4}"),
                   F.to_date(F.regexp_extract(s, r"^(\d{1,2}/\d{1,2}/\d{4})", 1), "M/d/yyyy"))
             .when(s.rlike(r"^\d{6}$"), F.to_date(F.concat(s, F.lit("01")), "yyyyMMdd"))
             .when(s.rlike(r"^\d{4}-\d{2}$"), F.to_date(F.concat(s, F.lit("-01")), "yyyy-MM-dd")))

def real_date(p):
    """Drop placeholder dates (year <= 1900 or >= 2100, e.g. 0001-01-01 / 9999-12-31) for min/max."""
    return F.when(F.year(p).between(1901, 2099), p)


def ym_add(m, k):
    t = int(m[:4]) * 12 + int(m[5:7]) - 1 + k
    return f"{t // 12:04d}-{t % 12 + 1:02d}"

def snap(m):
    return int(ym_add(m, SHIFT).replace("-", ""))

def clean_pwr(c):
    s = F.trim(F.col(c).cast("string"))
    s = F.when(s.rlike(r"^[0-9]{1,10}$"), F.lpad(s, 10, "0"))
    return F.when(s != F.lit("0000000000"), s)

def by_month(table):
    try:
        pp = spark.sql(f"SHOW PARTITIONS {table}").toPandas().iloc[:, 0].astype(str)
        tks = {int(v.split("time_key=", 1)[1].split("/")[0]) for v in pp if "time_key=" in v}
    except Exception:
        tks = set()
    if not tks:
        tks = {int(r[0]) for r in spark.table(table).select("time_key").distinct().collect() if r[0] is not None}
    out = {}
    for t in tks:
        if 19000101 <= t <= 21001231:
            out[t // 100] = max(out.get(t // 100, 0), t)
    return out

def share(a, b):
    return fpct(a / b) if b else ""

show([f"<h3>Setup</h3><p>Spark {esc(spark.version)}</p>",
      tbl(pd.DataFrame([("Value check months", ", ".join(VALUE_MONTHS)), ("DHE check months", ", ".join(DHE_MONTHS))],
                       columns=["setting", "value"]))])


In [ ]:
# ============================================================
# 2. Value coverage: accounts with / without power id, billing hierarchy
# ============================================================
bal = spark.table(OFS_BAL)
BAL_TK = by_month(OFS_BAL)
VALS = {"billed": "tot_bdf_billed", "inv_bal": "avg_investable_bal", "analyzed_chrg": "analyzed_chrg", "avg_bal": "cor_tda_avg_bal"}
DIMS = [c for c in ("data_src_cd", "hier_level", "hier_leaf_node", "bill_child_of_bill_parent", "bill_flg") if c in bal.columns]
NAME = next((c for c in ("cust_name", "lead_cust_name") if c in bal.columns), None)
miss = [c for c in list(VALS.values()) + ["cust_pwr_id", "curnt_cust_dim_key", "cust_dim_key", "time_key"] if c not in bal.columns]
if miss:
    raise RuntimeError(f"{OFS_BAL} lacks {miss}")

def snap_df(m):
    tk = BAL_TK.get(snap(m))
    if tk is None:
        raise RuntimeError(f"No balance snapshot for analysis month {m}")
    return tk, (bal.filter(F.col("time_key") == tk)
                   .select(clean_pwr("cust_pwr_id").alias("pwr"), F.col("curnt_cust_dim_key").cast("long").alias("ccdk"),
                           F.col("cust_dim_key").cast("long").alias("cdk"),
                           *[F.coalesce(F.trim(F.col(c).cast("string")), F.lit("(null)")).alias(c) for c in DIMS],
                           *([F.col(NAME).alias("name")] if NAME else []),
                           *[F.col(s).cast("double").alias(d) for d, s in VALS.items()])
                   .withColumn("has_pwr", F.when(F.col("pwr").isNotNull(), "Y").otherwise("N")))

# (a) by power id, all months
rows = []
for m in VALUE_MONTHS:
    tk, b = snap_df(m)
    g = b.groupBy("has_pwr").agg(F.count(F.lit(1)).alias("n"), *[F.sum(d).alias(d) for d in VALS]).toPandas().set_index("has_pwr")
    tot = g.sum()
    r = {"analysis month": m, "snapshot": tk, "accounts": f"{int(tot['n']):,}",
         "accounts without power id": share(g["n"].get("N", 0), tot["n"])}
    for d in VALS:
        r[f"{d} total"] = fnum(tot[d])
        r[f"{d} without power id"] = share(g[d].get("N", 0), tot[d])
    rows.append(r)
A = pd.DataFrame(rows)

# (b) latest value month: power id x hierarchy columns
mL = VALUE_MONTHS[-1]
tkL, bL = snap_df(mL)
bL = bL.persist(StorageLevel.MEMORY_AND_DISK)
totL = bL.agg(*[F.sum(d).alias(d) for d in VALS], F.count(F.lit(1)).alias("n")).collect()[0]
gb = (bL.groupBy("has_pwr", *DIMS).agg(F.count(F.lit(1)).alias("accounts"), *[F.sum(d).alias(d) for d in VALS])
         .orderBy(F.col("billed").desc_nulls_last()).limit(25).toPandas())
for d in VALS:
    gb[f"{d} share"] = gb[d].map(lambda v: share(v or 0, totL[d]))
    gb[d] = gb[d].map(fnum)
gb["accounts"] = gb["accounts"].map(lambda v: f"{int(v):,}")

# (c) can accounts without power id be linked?
nop = bL.filter(F.col("pwr").isNull())
nop_tot = nop.agg(F.count(F.lit(1)).alias("n"), F.sum("billed").alias("billed"), F.sum("inv_bal").alias("inv")).collect()[0]
def key_map(df, key):
    return (df.filter(F.col(key) > 0).groupBy(F.col(key).alias("_k"))
              .agg(F.countDistinct("pwr").alias("_np"), F.max("pwr").alias("_pm"))
              .filter("_np = 1").select("_k", F.col("_pm").alias("pwr_link")))
routes = [("current customer key -> other accounts in the same snapshot", "ccdk", key_map(bL.filter(F.col("pwr").isNotNull()), "ccdk")),
          ("customer key -> other accounts in the same snapshot", "cdk", key_map(bL.filter(F.col("pwr").isNotNull()), "cdk"))]
dim_note = ""
try:
    dd = spark.table(CUST_DIM)
    pc = next((c for c in ("cust_pwr_id", "customer_pwr_id", "pwr_id") if c in dd.columns), None)
    if "cust_dim_key" in dd.columns and pc:
        dm = key_map(dd.select(F.col("cust_dim_key").cast("long").alias("k"), clean_pwr(pc).alias("pwr")), "k")
        routes += [("customer key -> customer dimension", "cdk", dm), ("current customer key -> customer dimension", "ccdk", dm)]
    else:
        dim_note = f"{CUST_DIM} lacks cust_dim_key or a power id column"
except Exception as e:
    dim_note = f"cannot read {CUST_DIM}: {str(e).splitlines()[0][:150]}"
lrows = []
for name, key, mp in routes:
    r = (nop.join(mp, F.col(key) == F.col("_k"), "left")
            .agg(F.sum(F.col("pwr_link").isNotNull().cast("int")).alias("n"),
                 F.sum(F.when(F.col("pwr_link").isNotNull(), F.col("billed"))).alias("billed"),
                 F.sum(F.when(F.col("pwr_link").isNotNull(), F.col("inv_bal"))).alias("inv")).collect()[0])
    lrows.append({"route": name, "accounts linked": share(r["n"] or 0, nop_tot["n"]),
                  "billed linked": share(r["billed"] or 0, nop_tot["billed"]),
                  "investable balance linked": share(r["inv"] or 0, nop_tot["inv"])})
LK = pd.DataFrame(lrows)

# (d) largest accounts without power id
top = (nop.orderBy(F.col("billed").desc_nulls_last()).limit(TOP_N)
          .select(*(["name"] if NAME else []), *DIMS, "ccdk", "cdk", *VALS).toPandas())
for d in VALS:
    top[d] = top[d].map(fnum)

A.to_csv(os.path.join(OUT_DIR, "value_by_power_id.csv"), index=False)
show(["<h3>1. Value coverage</h3>",
      "<h4>Share of source totals on accounts WITHOUT a power id</h4>", tbl(A),
      f"<h4>{esc(mL)} (snapshot {tkL}): power id x billing hierarchy, 25 largest groups by billed</h4>", tbl(gb),
      "<p class='note'>If parent rows (hier_leaf_node = N, low hier_level) carry billed / balances that their children also "
      "carry, the source total double-counts and the customer table is not missing money.</p>",
      f"<h4>Can the {int(nop_tot['n']):,} accounts without a power id be linked? ({esc(mL)})</h4>", tbl(LK)]
     + ([f"<p class='warn'>{esc(dim_note)}</p>"] if dim_note else [])
     + [f"<h4>{TOP_N} largest accounts without a power id (by billed)</h4>", tbl(top)])
bL.unpersist()


In [ ]:
# ============================================================
# 3. DHE doubling: customer-fact loads on the month-end date
# ============================================================
fact = spark.table(CUST_FACT)
srows, lrows = [], []
for m in DHE_MONTHS:
    ym = int(m.replace("-", ""))
    mf = fact.filter(F.col("as_of_dt").between(ym * 100 + 1, ym * 100 + 31))
    a = mf.agg(F.max("as_of_dt")).collect()[0][0]
    if a is None:
        srows.append({"month": m, "as_of_dt": "(none)"})
        continue
    d = (mf.filter(F.col("as_of_dt") == a)
           .select("time_key", F.col("cust_dim_key").cast("long").alias("cdk"), clean_pwr("cust_pwr_id").alias("pwr"),
                   F.col("dhe").cast("double").alias("dhe"))
           .persist(StorageLevel.MEMORY_AND_DISK))
    lt = (d.groupBy("time_key").agg(F.count(F.lit(1)).alias("rows"), F.countDistinct("cdk").alias("cust_dim_keys"),
                                     F.countDistinct("pwr").alias("power_ids"), F.sum("dhe").alias("dhe"))
           .orderBy("time_key").toPandas())
    big = lt.sort_values("rows", ascending=False).head(4).sort_values("time_key")
    for _, x in big.iterrows():
        lrows.append({"month": m, "as_of_dt": a, "time_key": int(x["time_key"]), "rows": f"{int(x['rows']):,}",
                      "cust_dim_keys": f"{int(x['cust_dim_keys']):,}", "power ids": f"{int(x['power_ids']):,}",
                      "dhe": fnum(x["dhe"])})
    # current rule: latest version per cust_dim_key
    A_ = (d.withColumn("_r", F.row_number().over(Window.partitionBy("cdk").orderBy(F.col("time_key").desc())))
           .filter("_r = 1").filter(F.col("pwr").isNotNull()))
    # proposed rule: per power id, all rows of its latest load
    B_ = (d.filter(F.col("pwr").isNotNull())
           .withColumn("_t", F.max("time_key").over(Window.partitionBy("pwr"))).filter(F.col("time_key") == F.col("_t")))
    ra = A_.agg(F.count(F.lit(1)).alias("n"), F.countDistinct("pwr").alias("p"), F.sum("dhe").alias("dhe")).collect()[0]
    rb = B_.agg(F.count(F.lit(1)).alias("n"), F.countDistinct("pwr").alias("p"), F.sum("dhe").alias("dhe")).collect()[0]
    multi_a = A_.groupBy("pwr").agg(F.countDistinct("time_key").alias("k")).filter("k > 1").count()
    srows.append({"month": m, "as_of_dt": a, "loads": len(lt), "largest load rows": f"{int(lt['rows'].max()):,}",
                  "DHE current rule": fnum(ra["dhe"]), "DHE proposed rule": fnum(rb["dhe"]),
                  "rows per power id (current)": f"{ra['n'] / ra['p']:.2f}" if ra["p"] else "",
                  "rows per power id (proposed)": f"{rb['n'] / rb['p']:.2f}" if rb["p"] else "",
                  "power ids mixing 2+ loads (current)": f"{multi_a:,}"})
    d.unpersist()
S = pd.DataFrame(srows)
S.to_csv(os.path.join(OUT_DIR, "dhe_rules.csv"), index=False)
show(["<h3>2. DHE: current rule vs proposed rule</h3>", tbl(S),
      "<p class='note'>Current = latest version per cust_dim_key (as built). Proposed = per power id, all rows of its "
      "latest load. Power ids mixing 2+ loads under the current rule are the ones whose amounts can be counted twice.</p>",
      "<h4>Largest loads on each month-end date (up to 4)</h4>", tbl(pd.DataFrame(lrows))])
